In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_rfm_segments
# PURPOSE    : RFM customer segmentation
# SOURCE     : silver.orders, silver.order_payments
# DESTINATION: fincompliance_catalog.gold.rfm_segments
# METHODOLOGY:
#   R = Recency   : days since last order
#   F = Frequency : number of orders placed
#   M = Monetary  : total amount spent
#   Each scored 1-5 using ntile (quintile ranking)
#   Combined into customer segments:
#   Champions, Loyal, Potential, At Risk, Lost
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    max as spark_max,
    avg,
    round as spark_round,
    datediff,
    lit,
    when,
    current_timestamp,
    ntile,
    desc
)
from pyspark.sql.window import Window


In [0]:
# ============================================================
# READ FROM SILVER
# ============================================================

df_orders = spark.table(f"{SILVER_DB}.orders")
df_payments = spark.table(f"{SILVER_DB}.order_payments")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"orders         : {df_orders.count():,} rows")
print(f"order_payments : {df_payments.count():,} rows")

print("\nOnly using DELIVERED orders for RFM")
delivered = df_orders \
    .filter(col("order_status") == "delivered") \
    .count()
print(f"Delivered orders : {delivered:,}")

In [0]:
# ============================================================
# STEP 1 - CALCULATE RAW RFM VALUES
# For each customer calculate:
# R = days since last order
# F = total number of orders
# M = total amount spent
# ============================================================

# Reference date = most recent order date in dataset
reference_date = df_orders \
    .filter(col("order_status") == "delivered") \
    .agg(spark_max("order_purchase_timestamp")) \
    .collect()[0][0]

print(f"Reference date : {reference_date}")
print("(Most recent order date in dataset)")

# Calculate RFM raw values
df_rfm_raw = (
    df_orders
    .filter(col("order_status") == "delivered")
    .join(
        df_payments.select("order_id", "payment_value"),
        on="order_id",
        how="left"
    )
    .groupBy("customer_id")
    .agg(
        datediff(
            lit(reference_date),
            spark_max("order_purchase_timestamp")
        ).alias("recency_days"),
        count("order_id").alias("frequency"),
        spark_round(
            spark_sum("payment_value"), 2
        ).alias("monetary")
    )
)

print(f"\nTotal customers : {df_rfm_raw.count():,}")
print("\nRFM raw values sample (5 rows):")
df_rfm_raw.show(5, truncate=False)

print("\nRFM statistics:")
df_rfm_raw \
    .select("recency_days", "frequency", "monetary") \
    .summary("min", "max", "mean") \
    .show()

In [0]:
# ============================================================
# STEP 2 - SCORE EACH CUSTOMER 1-5 USING NTILE
# ntile(5) divides customers into 5 equal groups
# R score: LOW recency days = HIGH score (ordered recently)
# F score: HIGH frequency = HIGH score (orders often)
# M score: HIGH monetary = HIGH score (spends more)
# ============================================================

# Window specifications for each dimension
window_r = Window.orderBy(desc("recency_days"))
window_f = Window.orderBy("frequency")
window_m = Window.orderBy("monetary")

df_rfm_scored = df_rfm_raw \
    .withColumn(
        "r_score",
        ntile(5).over(window_r)
    ) \
    .withColumn(
        "f_score",
        ntile(5).over(window_f)
    ) \
    .withColumn(
        "m_score",
        ntile(5).over(window_m)
    ) \
    .withColumn(
        "rfm_score",
        col("r_score") +
        col("f_score") +
        col("m_score")
    )

print("RFM scored sample (10 rows):")
df_rfm_scored \
    .select(
        "customer_id",
        "recency_days",
        "frequency",
        "monetary",
        "r_score",
        "f_score",
        "m_score",
        "rfm_score"
    ) \
    .show(10, truncate=True)

print("\nRFM score distribution:")
df_rfm_scored \
    .groupBy("rfm_score") \
    .count() \
    .orderBy("rfm_score") \
    .show(20, truncate=False)

In [0]:
# ============================================================
# STEP 3 - HANDLE NULL MONETARY VALUES
# Some customers have null monetary
# This happens when payment record is missing
# Replace null monetary with 0 and rescore
# ============================================================

null_monetary = df_rfm_scored \
    .filter(col("monetary").isNull()) \
    .count()
print(f"Customers with null monetary : {null_monetary:,}")

# Replace null monetary with 0
df_rfm_scored = df_rfm_scored \
    .withColumn(
        "monetary",
        when(col("monetary").isNull(), 0)
        .otherwise(col("monetary"))
    ) \
    .withColumn(
        "m_score",
        when(col("monetary") == 0, 1)
        .otherwise(col("m_score"))
    )

print("Null monetary handled")
print(f"Total customers : {df_rfm_scored.count():,}")

In [0]:
# ============================================================
# STEP 4 - CLASSIFY CUSTOMERS INTO SEGMENTS
# Based on rfm_score total (3 to 15)
# Champions      : score 13-15 (best customers)
# Loyal          : score 10-12 (regular buyers)
# Potential      : score 7-9  (showing promise)
# At Risk        : score 5-6  (declining activity)
# Lost           : score 3-4  (inactive customers)
# ============================================================

df_rfm_segments = df_rfm_scored \
    .withColumn(
        "customer_segment",
        when(col("rfm_score") >= 13, "Champions")
        .when(col("rfm_score") >= 10, "Loyal")
        .when(col("rfm_score") >= 7,  "Potential")
        .when(col("rfm_score") >= 5,  "At Risk")
        .otherwise("Lost")
    )

# Show segment distribution
print("Customer segment distribution:")
df_rfm_segments \
    .groupBy("customer_segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Show segment statistics
print("\nSegment statistics:")
df_rfm_segments \
    .groupBy("customer_segment") \
    .agg(
        count("customer_id").alias("customer_count"),
        spark_round(avg("recency_days"), 1)
            .alias("avg_recency_days"),
        spark_round(avg("frequency"), 2)
            .alias("avg_frequency"),
        spark_round(avg("monetary"), 2)
            .alias("avg_monetary")
    ) \
    .orderBy("avg_monetary", ascending=False) \
    .show(truncate=False)

total = df_rfm_segments.count()
print(f"\nTotal customers segmented : {total:,}")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_rfm_segments = df_rfm_segments \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_rfm_segments.columns:
    print(f"  {col_name}")
print(f"Total customers : {df_rfm_segments.count():,}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_rfm_segments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.rfm_segments")
)

print(f"Written to {GOLD_DB}.rfm_segments")
print(f"Rows : {df_rfm_segments.count():,}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.RFM_SEGMENTS VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.rfm_segments")
print(f"Total customers : {df_gold.count():,}")
print(f"Total columns   : {len(df_gold.columns)}")

print("\nSegment distribution:")
df_gold \
    .groupBy("customer_segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\nSegment revenue potential:")
df_gold \
    .groupBy("customer_segment") \
    .agg(
        count("customer_id").alias("customers"),
        spark_round(
            spark_sum("monetary"), 2
        ).alias("total_revenue"),
        spark_round(
            avg("monetary"), 2
        ).alias("avg_revenue_per_customer")
    ) \
    .orderBy("total_revenue", ascending=False) \
    .show(truncate=False)

print("=" * 60)
print("gold.rfm_segments verification complete!")
print("=" * 60)